In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

import keras.backend as K

from keras.utils.vis_utils import plot_model

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [ ]:
# Try to make GPU memory allocation dynamic, so it doesn't slow down/crash
physical_devices = tf.config.list_physical_devices('GPU')
try:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # config = tf.compat.v1.ConfigProto()
    # config.gpu_options.allow_growth = True
    # session = tf.compat.v1.Session(config=config)
    tf.debugging.set_log_device_placement(False)
except:
    print('oh no.')
  # Invalid device or cannot modify virtual devices once initialized.
    pass

In [ ]:
pip install opencv-python

In [ ]:
pip install imutils

In [ ]:
pip install visualkeras

In [ ]:
pip install pydot

In [ ]:
from tqdm import tqdm
import cv2
# import os
import imutils

# change this to wherever your images are located
DATASET_PATH = r'C:/Users/Aihara/Desktop/Temporary Jupter Stuff/Kaggle - Brain Tumor MRI Dataset'


def crop_img(img):
	"""
	Finds the extreme points on the image and crops the rectangular out of them
	"""
	gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
	gray = cv2.GaussianBlur(gray, (3, 3), 0)

	# threshold the image, then perform a series of erosions +
	# dilations to remove any small regions of noise
	thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
	thresh = cv2.erode(thresh, None, iterations=2)
	thresh = cv2.dilate(thresh, None, iterations=2)

	# find contours in thresholded image, then grab the largest one
	cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
	cnts = imutils.grab_contours(cnts)
	c = max(cnts, key=cv2.contourArea)

	# find the extreme points
	extLeft = tuple(c[c[:, :, 0].argmin()][0])
	extRight = tuple(c[c[:, :, 0].argmax()][0])
	extTop = tuple(c[c[:, :, 1].argmin()][0])
	extBot = tuple(c[c[:, :, 1].argmax()][0])
	ADD_PIXELS = 0
	new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()
	
	return new_img
	
if __name__ == "__main__":
	training = DATASET_PATH + "/Training"
	testing = DATASET_PATH + "/Testing"
	training_dir = os.listdir(training)
	testing_dir = os.listdir(testing)
	IMG_SIZE = 256

	for dir in training_dir:
		save_path = 'cleaned/Training/'+ dir
		path = os.path.join(training,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)
	
	for dir in testing_dir:
		save_path = 'cleaned/Testing/'+ dir
		path = os.path.join(testing,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)

In [ ]:
# Code to load images borrowed from https://github.com/masoudnick/Brain-Tumor-MRI-Classification/blob/main/Brain_Tumor_Classification.ipynb
# We'll adjust this code to also load in the generated images.

# Change these to where your cleaned data are located
CLEANED_TRAINING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Training"
CLEANED_TESTING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Testing"
GEN_IMAGE_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\Generated"


labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

x_train = [] # training images.
y_test  = [] # training labels.
image_size = 200



for label in labels:
    trainPath = os.path.join(CLEANED_TRAINING_PATH,label)
    for file in tqdm(os.listdir(trainPath)):
        image = cv2.imread(os.path.join(trainPath, file),0) # load images in gray.
        image = cv2.bilateralFilter(image, 2, 50, 50) # remove images noise.
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
        image = cv2.resize(image, (image_size, image_size)) # resize images into 200*200.
        x_train.append(image)
        y_train.append(labels.index(label))
    
    testPath = os.path.join(CLEANED_TESTING_PATH,label)
    for file in tqdm(os.listdir(testPath)):
        image = cv2.imread(os.path.join(testPath, file),0)
        image = cv2.bilateralFilter(image, 2, 50, 50)
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
        image = cv2.resize(image, (image_size, image_size))
        x_test.append(image)
        y_test.append(labels.index(label))


# This will read in the generated images. No need to de-noise or resize them, since they were generated as 200*200. 

x_fake = []
for file in tqdm(os.listdir(GEN_IMAGE_PATH)):
    image = cv2.imread(os.path.join(trainPath, file),0) # load images in gray.
    image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
    x_fake.append(image)

    
x_train = np.array(x_train, dtype=np.float32) / 255.0 # normalize Images into range 0 to 1.
x_test = np.array(x_test, dtype=np.float32) / 255.0
x_fake = np.array(x_fake, dtype=np.float32) / 255.0

x_fake_train, x_fake_test = sklearn.model_selection.train_test_split(x_fake, test_size = 0.2)

print(x_real.shape)
print(x_fake.shape)

In [ ]:
# Now we will implement the evaluation metric. The simplest is GQI: (1) Train a classifier on real images
#                                                                   (2) Use this classifier to get labels for generated images
#                                                                   (3) Train a similar classifier on generated images
#                                                                   (4) Determine accuracy ratio of the two classifiers.
#
# If possible, it would be better to calculate FID instead.

tf.random.set_seed(42)
np.random.seed(42)



# CNN for classifying the real images. Borrowed and adapted from HOA #7.
# We can use transfer learning to train the classifier for the fake images later.
BT_classify_real = keras.models.Sequential([
    DefaultConv2D(filters=64, kernel_size=5, input_shape=[200, 200, 3]),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=128),
    DefaultConv2D(filters=128),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=256),
    DefaultConv2D(filters=256),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=4, activation='softmax'),
])